In [1]:
# 1. Khai báo thư viện và đọc dữ liệu gốc từ data/process
import ast
import pandas as pd
import geopandas as gpd
from collections import Counter, defaultdict
from shapely.geometry import LineString

pd.set_option('display.max_columns', None)

gdf = gpd.read_file(r'C:\Tailieu\PPNC\data\data\process\ways.geojson')
df_ways = pd.read_csv(r'C:\Tailieu\PPNC\data\data\process\ways.csv')
df_nodes = pd.read_csv(r'C:\Tailieu\PPNC\data\data\process\nodes.csv')

print('Số đoạn ban đầu:', len(gdf), '| Số điểm (nodes) ban đầu:', len(df_nodes))

Số đoạn ban đầu: 1675 | Số điểm (nodes) ban đầu: 6845


In [2]:
# 2. Xử lý sai định dạng (Format)
gdf['nodes'] = df_ways['nodes'].apply(ast.literal_eval)

for col in ['name', 'highway', 'service', 'oneway']:
    gdf[col] = gdf[col].astype('string').str.strip()

gdf[['id', 'name', 'highway', 'service', 'oneway', 'nodes']].head(5)

,id,name,highway,service,oneway,nodes
0,32575790,Hẻm 31 Ung Văn Khiêm,residential,<NA>,<NA>,"[366414641, 5733448375, 2030638785, 2030639268..."
1,32575792,<NA>,residential,<NA>,<NA>,"[366452436, 5737751039, 366379167, 366404193, ..."
2,32575795,Đường D2,residential,<NA>,<NA>,"[5733055246, 366467734, 5881087223, 366442186,..."
3,32575984,Mê Linh,residential,<NA>,<NA>,"[366368194, 5671815972, 366458884, 5671815966,..."
4,32576073,Võ Oanh,tertiary,<NA>,<NA>,"[366403574, 14302380595, 366386878, 1430269991..."


In [3]:
# 3. Xử lý dữ liệu trùng lặp và hình học lỗi (Duplicates & Invalid)
gdf = gdf.drop_duplicates(subset=['id']).copy()
df_nodes = df_nodes.drop_duplicates(subset=['id']).copy()

gdf = gdf[gdf['geometry'].notna() & (~gdf['geometry'].is_empty)].copy()
gdf = gdf[gdf['nodes'].apply(len) >= 2].reset_index(drop=True)

gdf['start_node'] = gdf['nodes'].apply(lambda x: x[0])
gdf['end_node'] = gdf['nodes'].apply(lambda x: x[-1])

print('Số đoạn hợp lệ:', len(gdf), '| Số điểm (nodes) hợp lệ:', len(df_nodes))

Số đoạn hợp lệ: 1675 | Số điểm (nodes) hợp lệ: 6845


In [4]:
# 4. Phân loại Đường chính, Hẻm cấp 1, Hẻm cấp 2, Hẻm cấp 3+, Hẻm không tên
def get_road_type(row):
    name = '' if pd.isna(row['name']) else str(row['name']).strip()
    highway = '' if pd.isna(row['highway']) else str(row['highway']).strip()
    
    if highway in ['footway', 'steps', 'pedestrian', 'path', 'cycleway', 'corridor']:
        return 'Đường đi bộ'
    if 'hẻm' in name.lower():
        slash = name.count('/')
        if slash == 0:
            return 'Hẻm cấp 1'
        elif slash == 1:
            return 'Hẻm cấp 2'
        else:
            return 'Hẻm cấp 3+'
    if highway in ['trunk', 'trunk_link', 'primary', 'primary_link', 'secondary', 'secondary_link', 'tertiary', 'tertiary_link'] or name != '':
        return 'Đường chính'
    return 'Hẻm không tên'

gdf['road_type'] = gdf.apply(get_road_type, axis=1)
gdf['is_alley'] = gdf['road_type'].str.startswith('Hẻm')

print(gdf['road_type'].value_counts())

road_type
Đường đi bộ      390
Đường chính      348
Hẻm cấp 1        346
Hẻm không tên    310
Hẻm cấp 2        257
Hẻm cấp 3+        24
Name: count, dtype: int64


In [5]:
# 5. Xử lý dữ liệu rỗng (Giữ lại các Hẻm không tên, xóa các cột trống >85% như maxspeed, lanes, width...)
gdf['name'] = gdf['name'].fillna(gdf['is_alley'].map({True: 'Hẻm không tên', False: 'Đường không tên'}))
gdf['oneway'] = gdf['oneway'].fillna('no')

keep_cols = ['id', 'name', 'highway', 'road_type', 'is_alley', 'oneway', 'nodes', 'start_node', 'end_node', 'geometry']
gdf = gdf[keep_cols].copy()

print(gdf.isnull().sum())

id            0
name          0
highway       0
road_type     0
is_alley      0
oneway        0
nodes         0
start_node    0
end_node      0
geometry      0
dtype: int64


In [6]:
# 6. Tính độ dài thật từ tọa độ bản đồ (length - đơn vị mét)
gdf['length'] = gdf.to_crs(epsg=32648).geometry.length.round(2)

gdf[['id', 'name', 'highway', 'road_type', 'is_alley', 'oneway', 'length']].head(10)

,id,name,highway,road_type,is_alley,oneway,length
0,32575790,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,no,237.62
1,32575792,Hẻm không tên,residential,Hẻm không tên,True,no,606.37
2,32575795,Đường D2,residential,Đường chính,False,no,249.66
3,32575984,Mê Linh,residential,Đường chính,False,no,228.97
4,32576073,Võ Oanh,tertiary,Đường chính,False,no,467.43
5,32576539,Đường không tên,primary_link,Đường chính,False,no,8.55
6,32577627,Huỳnh Mẫn Đạt,residential,Đường chính,False,no,192.08
7,32577688,Xô Viết Nghệ Tĩnh,primary,Đường chính,False,yes,41.70
8,32578249,Đường D5,tertiary,Đường chính,False,no,164.64
9,32578410,Hẻm 362 Ung Văn Khiêm,residential,Hẻm cấp 1,True,no,191.87


In [7]:
# 7. Đếm số nhánh rẽ (degree) tại mỗi điểm node
node_way_count = Counter()
degree_count = Counter()
node_roads = defaultdict(set)

for _, row in gdf.iterrows():
    node_ids = row['nodes']
    for i, nid in enumerate(node_ids):
        node_way_count[nid] += 1
        node_roads[nid].add(row['name'])
        if 0 < i < len(node_ids) - 1:
            degree_count[nid] += 2
        else:
            degree_count[nid] += 1

print('Tổng số node đã tính bậc:', len(degree_count))

Tổng số node đã tính bậc: 6845


In [8]:
# 8. Tính Trạng thái thông/cụt (alley_status), Số ngã rẽ (turn_count), Số nút giao (intersection_count) và Đường giao nhau
alley_status = []
turn_count = []
intersection_count = []
connected_roads = []

for _, row in gdf.iterrows():
    node_ids = row['nodes']
    u, v = row['start_node'], row['end_node']
    is_dead_end = (degree_count[u] == 1 or degree_count[v] == 1)
    
    if row['is_alley']:
        alley_status.append('Hẻm cụt' if is_dead_end else 'Hẻm thông')
    else:
        alley_status.append('Đường cụt' if is_dead_end else 'Đường thông')
        
    turns = [nid for nid in node_ids if degree_count[nid] >= 3]
    inters = [nid for nid in node_ids if node_way_count[nid] > 1]
    turn_count.append(len(turns))
    intersection_count.append(len(inters))
    
    others = {r for nid in inters for r in node_roads[nid] if r != row['name']}
    connected_roads.append(', '.join(sorted(others)) if others else 'Không có')

gdf['alley_status'] = alley_status
gdf['turn_count'] = turn_count
gdf['intersection_count'] = intersection_count
gdf['connected_roads'] = connected_roads

gdf_alley = gdf[gdf['is_alley'] == True].copy()
gdf[['id', 'name', 'road_type', 'alley_status', 'length', 'turn_count', 'intersection_count', 'connected_roads']].head(10)

,id,name,road_type,alley_status,length,turn_count,intersection_count,connected_roads
0,32575790,Hẻm 31 Ung Văn Khiêm,Hẻm cấp 1,Hẻm thông,237.62,8,8,"Hẻm 31/15 Ung Văn Khiêm, Hẻm 31/2 Ung Văn Khiê..."
1,32575792,Hẻm không tên,Hẻm không tên,Hẻm cụt,606.37,5,5,"Hẻm 860 Xô Viết Nghệ Tĩnh, Hẻm 860/60X Xô Viết..."
2,32575795,Đường D2,Đường chính,Đường thông,249.66,3,3,"Hẻm 860/60X Xô Viết Nghệ Tĩnh, Hẻm không tên, ..."
3,32575984,Mê Linh,Đường chính,Đường thông,228.97,7,7,"Hẻm 17 Mê Linh, Hẻm 32 Đường Mê Linh, Hẻm 35 M..."
4,32576073,Võ Oanh,Đường chính,Đường thông,467.43,9,9,"Hẻm 1 Võ Oanh, Hẻm 24 Võ Oanh, Hẻm 36 Nguyễn G..."
5,32576539,Đường không tên,Đường chính,Đường cụt,8.55,1,1,"Hẻm 220 Xô Viết Nghệ Tĩnh, Xô Viết Nghệ Tĩnh"
6,32577627,Huỳnh Mẫn Đạt,Đường chính,Đường thông,192.08,6,6,"Huỳnh Tịnh Của, Hẻm 23 Huỳnh Mẫn Đạt, Hẻm 37 H..."
7,32577688,Xô Viết Nghệ Tĩnh,Đường chính,Đường thông,41.70,2,3,"Cầu Thị Nghè 1, Phan Văn Hân"
8,32578249,Đường D5,Đường chính,Đường thông,164.64,3,3,"Hẻm 4 Đường D5, Hẻm 42 Ung Văn Khiêm, Hẻm khôn..."
9,32578410,Hẻm 362 Ung Văn Khiêm,Hẻm cấp 1,Hẻm thông,191.87,5,5,"Hẻm 38 Tân Cảng, Hẻm 667 Điện Biên Phủ, Hẻm 67..."


In [9]:
# 9. Tạo bảng tham số Nút giao thông (df_nodes_clean)
node_dict = dict(zip(df_nodes['id'], zip(df_nodes['lon'], df_nodes['lat'])))

graph_nodes = set()
for node_ids in gdf['nodes']:
    graph_nodes.add(node_ids[0])
    graph_nodes.add(node_ids[-1])
    for nid in node_ids[1:-1]:
        if node_way_count[nid] > 1:
            graph_nodes.add(nid)

node_rows = []
for nid in sorted(graph_nodes):
    if nid not in node_dict:
        continue
    lon, lat = node_dict[nid]
    deg = degree_count[nid]
    node_type = 'Nút giao' if deg >= 3 else ('Điểm cụt' if deg == 1 else 'Điểm nối')
    
    node_rows.append({
        'id': nid,
        'lat': lat,
        'lon': lon,
        'degree': deg,
        'node_type': node_type,
        'connected_roads': ', '.join(sorted(node_roads[nid]))
    })

df_nodes_clean = pd.DataFrame(node_rows)
df_nodes_clean.head(10)

,id,lat,lon,degree,node_type,connected_roads
0,366367285,10.804989,106.721200,3,Nút giao,"Hẻm 143A Ung Văn Khiêm, Ung Văn Khiêm"
1,366368194,10.790241,106.709544,3,Nút giao,"Mê Linh, Phạm Viết Chánh"
2,366368684,10.807179,106.718013,3,Nút giao,"Hẻm 103 Đường Ung Văn Khiêm, Ung Văn Khiêm"
3,366372116,10.792385,106.708110,5,Nút giao,"Công trường Hòa Bình, Hẻm 33 Phan Văn Hân, Hẻm..."
4,366372456,10.799902,106.720995,3,Nút giao,"Hẻm 38 Tân Cảng, Tân Cảng"
5,366376129,10.807628,106.716367,4,Nút giao,"Nguyễn Gia Trí, Ung Văn Khiêm, Đường không tên"
6,366377224,10.797382,106.710784,3,Nút giao,"Hẻm 160 Xô Viết Nghệ Tĩnh, Xô Viết Nghệ Tĩnh"
7,366377501,10.802866,106.719724,3,Nút giao,"Hẻm 132 Nguyễn Văn Thương, Nguyễn Văn Thương"
8,366380192,10.792710,106.710348,3,Nút giao,"Hẻm 82 Nguyễn Công Trứ, Đường Nguyễn Công Trứ"
9,366380695,10.803225,106.716392,3,Nút giao,"Hẻm 36 Nguyễn Gia Trí, Hẻm không tên"


In [10]:
# 10. Tạo bảng tham số Cạnh đồ thị (df_edges_clean) phục vụ chạy thuật toán
edge_rows = []

for _, row in gdf.iterrows():
    node_ids = [nid for nid in row['nodes'] if nid in node_dict]
    split_idx = [0] + [i for i in range(1, len(node_ids) - 1) if node_ids[i] in graph_nodes] + [len(node_ids) - 1]
    
    for i in range(len(split_idx) - 1):
        sub_nodes = node_ids[split_idx[i] : split_idx[i+1] + 1]
        if len(sub_nodes) < 2:
            continue
        coords = [node_dict[nid] for nid in sub_nodes]
        edge_rows.append({
            'way_id': row['id'],
            'u': sub_nodes[0],
            'v': sub_nodes[-1],
            'name': row['name'],
            'highway': row['highway'],
            'road_type': row['road_type'],
            'is_alley': row['is_alley'],
            'alley_status': row['alley_status'],
            'oneway': row['oneway'],
            'geometry': LineString(coords)
        })

gdf_edges = gpd.GeoDataFrame(edge_rows, geometry='geometry', crs='EPSG:4326')
gdf_edges['edge_id'] = range(1, len(gdf_edges) + 1)
gdf_edges['length'] = gdf_edges.to_crs(epsg=32648).geometry.length.round(2)

df_edges_clean = gdf_edges[['edge_id', 'way_id', 'u', 'v', 'name', 'highway', 'road_type', 'is_alley', 'alley_status', 'oneway', 'length']].copy()
df_edges_clean.head(10)

,edge_id,way_id,u,v,name,highway,road_type,is_alley,alley_status,oneway,length
0,1,32575790,366414641,5733448375,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,67.50
1,2,32575790,5733448375,2030638785,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,7.84
2,3,32575790,2030638785,2030639268,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,46.39
3,4,32575790,2030639268,2030639086,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,26.60
4,5,32575790,2030639086,5737826545,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,17.39
5,6,32575790,5737826545,2030638786,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,41.09
6,7,32575790,2030638786,366404504,Hẻm 31 Ung Văn Khiêm,residential,Hẻm cấp 1,True,Hẻm thông,no,30.82
7,8,32575792,366452436,5737751039,Hẻm không tên,residential,Hẻm không tên,True,Hẻm cụt,no,35.90
8,9,32575792,5737751039,366390786,Hẻm không tên,residential,Hẻm không tên,True,Hẻm cụt,no,131.46
9,10,32575792,366390786,1516645265,Hẻm không tên,residential,Hẻm không tên,True,Hẻm cụt,no,136.65


In [11]:
# 11. Xuất 4 bảng dữ liệu sạch ra CSV
output_dir = r'C:\Tailieu\PPNC\data\data\process'

df_roads_clean = gdf.drop(columns=['geometry', 'nodes']).copy()
df_alleys_clean = gdf_alley.drop(columns=['geometry', 'nodes']).copy()

df_roads_clean.to_csv(f'{output_dir}\\roads_clean.csv', index=False, encoding='utf-8-sig')
df_alleys_clean.to_csv(f'{output_dir}\\alleys_clean.csv', index=False, encoding='utf-8-sig')
df_nodes_clean.to_csv(f'{output_dir}\\nodes_clean.csv', index=False, encoding='utf-8-sig')
df_edges_clean.to_csv(f'{output_dir}\\edges_clean.csv', index=False, encoding='utf-8-sig')

print('Đã lưu 4 bảng dữ liệu (giữ nguyên cả các Hẻm không tên, không fill số giả định):')
print('- roads_clean.csv :', len(df_roads_clean), 'đoạn đường & hẻm')
print('- alleys_clean.csv:', len(df_alleys_clean), 'đoạn hẻm (gồm cả Hẻm không tên)')
print('- nodes_clean.csv :', len(df_nodes_clean), 'nút giao thông')
print('- edges_clean.csv :', len(df_edges_clean), 'cạnh đồ thị')

Đã lưu 4 bảng dữ liệu (giữ nguyên cả các Hẻm không tên, không fill số giả định):
- roads_clean.csv : 1675 đoạn đường & hẻm
- alleys_clean.csv: 937 đoạn hẻm (gồm cả Hẻm không tên)
- nodes_clean.csv : 2615 nút giao thông
- edges_clean.csv : 3416 cạnh đồ thị
